In [1]:
from inu.utils.filesproc import Locator
from pathlib import Path
import os

root = Path().cwd()
os.environ['CUSTOM_OS_ROOT'] = str(Path().home())

# Locator

In [2]:
Locator?

Init signature:
Locator(
    *folders: 'PathT | Locator',
    envar: 'str' = None,
    sub: 'PathT' = None,
    order='EIA',
)
Docstring:     
Manages multiple locations in the file system and provides methods to find and valudate them.

Each of the locations can be defined:
 - explicitely as a path,
 - as environment variable
 - as another Locator

Main functionality resides in ``Locator.existing_folders()`` method.
By default it returns iterator over actually existing folders at the time of the call.
Init docstring:
Define locations to check for usually known before the run-time.

Locations are searched in order is defined by the string of following letters denoting:
  - *E*: folder from the Env variable
  - *I*: Internal fallback folders
  - *A*: Additional folders

Every letter must be used no more than once.
Valid orders: '*IA*', '*E*', '*AIE*'.

:param envar:       environment variable with search folder
:param default:     default search folder
:param sub:         sub-path relat

# Creation

## Single Argument

In [60]:
loc1 = Locator('/tmp', '~/tmp', envar='CUSTOM_OS_ROOT') # using environment variable
loc1

Locator<EIA>$CUSTOM_OS_ROOT=/home/itayo [/tmp
	/home/itayo/tmp]

In [65]:
loc2 = Locator(loc1, '/more/tmp') # using other Locator

## Search Order
Main functionality of the `Locator` class is iterate over provided locations:
 - **E**nvironment Variable based folder
 - **I**nternally defined list of folders
 - **A**dditionally provided to a specific method

Default: 'EIA'

*order* argument exists for almost all of the methods, and allows not only change the search order, but also filter it.  

### Iterators 
There are two family of iterators over the locations:
 - defined: does not checks if defined folders exist
 - existing: yields only existing 

In [66]:
[*loc1.defined()]

[PosixPath('/home/itayo'), PosixPath('/tmp'), PosixPath('/home/itayo/tmp')]

In [74]:
[*loc1.existing()]

[PosixPath('/home/itayo'), PosixPath('/tmp')]

In [75]:
[*loc1.existing(order='IE')]

[PosixPath('/tmp'), PosixPath('/home/itayo')]

Shortcuts for `first_` elements:

In [77]:
loc1.first_existing(), loc1.first_existing(order='AIE')

(PosixPath('/home/itayo'), PosixPath('/tmp'))

### Internal folders first, then environment variable

### Internal, then environment, then explicit

In [78]:
loc6 = Locator(root, envar='CUSTOM_OS_ROOT', order='IEA')

In [79]:
cnt = 1
for folder in loc6.existing(Path().home() / 'code'):
    print(f'{cnt}.{folder}')
    cnt += 1

1./home/itayo/code/algodev/inu/utils
2./home/itayo
3./home/itayo/code


# Finding Files

## Using Iterator

In [80]:
itr = loc5.find_file_iter('filesproc.py')
next(itr)

PosixPath('/home/itayo/code/algodev/inu/utils/filesproc.py')

## Using First Found

In [81]:
loc5.first_file('filesproc.py')

PosixPath('/home/itayo/code/algodev/inu/utils/filesproc.py')

# Defined and Existing

Defined locations inside the locator are not necessarily exists.  
So we distinguish between *defined* and *existing* iterators.

In [82]:
os.environ['FAKE_CUSTOM_OS_ROOT'] = str(Path().home() / 'notreallyexists')
loc7 = Locator(root, envar='FAKE_CUSTOM_OS_ROOT', order='IE')
loc7

Locator<IE>$FAKE_CUSTOM_OS_ROOT=/home/itayo/notreallyexists [/home/itayo/code/algodev/inu/utils]

In [83]:
[*loc7.defined()]

[PosixPath('/home/itayo/code/algodev/inu/utils'),
 PosixPath('/home/itayo/notreallyexists')]

In [84]:
[*loc7.existing()]

[PosixPath('/home/itayo/code/algodev/inu/utils')]

# Reconfigure existing Locator
For special cases, it is possible to reconfigure the Locator after it's creation.

In [85]:
loc7.reconfigure(root.parent, '/tmp', '/another/one', envar='CUSTOM_OS_ROOT')

In [86]:
loc7

Locator<IE>$CUSTOM_OS_ROOT=/home/itayo [/home/itayo/code/algodev/inu
	/tmp
	/another/one]

# Sub-foldering

In [87]:
loc4 = Locator(root, envar='CUSTOM_OS_ROOT')

In [88]:
loc8 = loc4 / 'code/algodev'

In [89]:
loc8

Locator<EIA|'/code/algodev'>$CUSTOM_OS_ROOT=/home/itayo [/home/itayo/code/algodev/inu/utils]

In [91]:
loc9 = loc8 / 'aviv'

In [92]:
loc9

Locator<EIA|'/code/algodev/aviv'>$CUSTOM_OS_ROOT=/home/itayo [/home/itayo/code/algodev/inu/utils]

In [90]:
loc8.sub

PosixPath('code/algodev')

# Operator overloading and dunder methods

## +

In [94]:
[dfnd for dfnd in (Locator(root) + ['/dev', '/aviv/']).defined()]

[PosixPath('/home/itayo/code/algodev/inu/utils'),
 PosixPath('/dev'),
 PosixPath('/aviv')]

## -

In [22]:
[dfnd for dfnd in (Locator(root) - root).defined()]

[]

## /

In [23]:
(Locator("/tmp", sub="one") / "two").sub == (Locator("/tmp", sub="one/two")).sub

True

## bool

In [24]:
bool(loc8)

True

In [25]:
bool(Locator())

False